# Local Neutral Atom Simulator Demo
## Qiskit & Cirq submission with qubit loss noise

This notebook demonstrates how to run quantum circuits on the local **NeutralAtomDevice** simulator using both Qiskit and Cirq.

Key features shown:
- Native gate decomposition (`Rz`, `SX`, `CZ`)
- Qubit loss noise modelling via `NoiseConfig`
- Separation of **accepted shots** (clean `{0,1}` results) from **raw shots** (includes qubit loss markers)
- Side-by-side histogram visualizations for both frameworks


## 1. Import Libraries

In [ ]:
# Qiskit
from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_histogram

# Qiskit neutral-atom backend
from qdk.qiskit import NeutralAtomBackend

# Cirq
import cirq

# Cirq neutral-atom sampler
from qdk.cirq import NeutralAtomSampler

# Shared noise configuration
from qdk.simulation import NoiseConfig

# Plotting
import matplotlib.pyplot as plt
from collections import Counter

print("Imports successful.")


## 2. Build the GHZ Circuit (Qiskit)

We construct a 3-qubit GHZ state circuit and then **transpile** it to the native gate set
supported by the NeutralAtomDevice: `{Rz, SX, CZ}`.

Transpiling first lets us run with `skip_transpilation=True`, which bypasses the backend's
own transpile step and gives us full control over the decomposition.


In [ ]:
n_qubits = 3

# High-level GHZ circuit
ghz_qiskit = QuantumCircuit(n_qubits, n_qubits)
ghz_qiskit.h(0)
for i in range(n_qubits - 1):
    ghz_qiskit.cx(i, i + 1)
ghz_qiskit.measure(range(n_qubits), range(n_qubits))

print("High-level circuit:")
print(ghz_qiskit.draw())

# Decompose to native gate set {Rz, SX, CZ}
backend = NeutralAtomBackend()
native_qiskit = transpile(
    ghz_qiskit,
    backend=backend
)

print("\nNative gate circuit:")
print(native_qiskit.draw())


## 3. Noiseless Simulation — Qiskit

Run the native circuit on the local simulator with **no noise**.
We expect to see only the two ideal GHZ outcomes: `000` and `111`.


In [ ]:
SHOTS = 500
SEED = 42

job_noiseless = backend.run(
    native_qiskit,
    shots=SHOTS,
    seed=SEED,
    skip_transpilation=True,
)
result_noiseless = job_noiseless.result()
data_noiseless = result_noiseless.results[0].data

print("Noiseless counts:", dict(data_noiseless.counts))
fig, ax = plt.subplots(figsize=(6, 4))
plot_histogram(dict(data_noiseless.counts), ax=ax)
ax.set_title("Qiskit — Noiseless GHZ")
ax.yaxis.grid(True, linestyle="--", alpha=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()


## 4. Noisy Simulation — Qiskit (8% Rz qubit loss)

Configure an 8% qubit-loss probability on `Rz` gates and re-run.

The `H` gate decomposes to `Rz` gates in the native set, so every qubit passes through
at least one `Rz` and has a chance of being lost.

**Accepted shots** (`counts`) exclude any shot where at least one qubit was lost.  
**Raw shots** (`raw_counts`) include all shots; lost qubits appear as `"-"` in the bitstring.


In [ ]:
noise = NoiseConfig()
noise.rz.loss = 0.08  # 8% qubit-loss probability per Rz gate

job_noisy = backend.run(
    native_qiskit,
    shots=SHOTS,
    noise=noise,
    seed=SEED,
    skip_transpilation=True,
)
result_noisy = job_noisy.result()
data_noisy = result_noisy.results[0].data

accepted = dict(data_noisy.counts)
raw      = dict(data_noisy.raw_counts)

total_accepted = sum(accepted.values())
total_raw      = sum(raw.values())
lost           = total_raw - total_accepted

print(f"Total shots : {total_raw}")
print(f"Accepted    : {total_accepted}  ({100*total_accepted/total_raw:.1f}%)")
print(f"Lost        : {lost}  ({100*lost/total_raw:.1f}%)")
print()
print("Accepted counts:", accepted)
print("Raw counts     :", raw)


## 5. Visualize Qiskit Results — Filtered vs Raw Histograms

Side-by-side comparison showing the accepted (clean) distribution on the left and the full
raw distribution (including loss bitstrings) on the right.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_histogram(accepted, ax=axes[0])
plot_histogram(raw,      ax=axes[1])

titles = [
    "Qiskit — Accepted shots (loss filtered out)",
    "Qiskit — Raw shots (loss bitstrings included)",
]
for ax, title in zip(axes, titles):
    ax.set_title(title)
    ax.yaxis.grid(True, linestyle="--", alpha=0.7)
    ax.set_axisbelow(True)

plt.tight_layout()
plt.show()


## 6. Build the GHZ Circuit (Cirq)

Cirq circuits work directly with `cirq.LineQubit` objects.  
The `NeutralAtomSampler` accepts standard Cirq circuits and internally converts them to
OpenQASM 3.0 before simulating.


In [ ]:
qubits = cirq.LineQubit.range(n_qubits)

ghz_cirq = cirq.Circuit(
    cirq.H(qubits[0]),
    *[cirq.CNOT(qubits[i], qubits[i + 1]) for i in range(n_qubits - 1)],
    cirq.measure(*qubits, key="result"),
)

print(ghz_cirq)


## 7. Noiseless Simulation — Cirq

Run with no noise and confirm only `000` and `111` appear.


In [ ]:
sampler = NeutralAtomSampler()

cirq_result_noiseless = sampler.run(ghz_cirq, repetitions=SHOTS)

# histogram() returns a Counter of integer bitmask → count
cirq_noiseless_counts = dict(cirq_result_noiseless.histogram(key="result"))

# Format integer keys as binary bitstrings for readability
cirq_noiseless_str = {format(k, f"0{n_qubits}b"): v for k, v in cirq_noiseless_counts.items()}
print("Noiseless Cirq counts:", cirq_noiseless_str)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(cirq_noiseless_str.keys(), cirq_noiseless_str.values())
ax.set_title("Cirq — Noiseless GHZ")
ax.set_xlabel("Outcome")
ax.set_ylabel("Count")
ax.yaxis.grid(True, linestyle="--", alpha=0.7)
ax.set_axisbelow(True)
for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        str(int(bar.get_height())),
        ha="center",
        va="bottom",
        fontsize=9,
    )
plt.tight_layout()
plt.show()


## 8. Noisy Simulation — Cirq (8% Rz qubit loss)

The same `NoiseConfig` is passed to the Cirq sampler.

- **`result.measurements["result"]`** — accepted shots only (int8 array, no loss rows)
- **`result.raw_measurements()["result"]`** — all shots; lost qubits are `"-"` strings
- **`result.raw_shots`** — raw simulator output as a list of `Result` enum tuples


In [ ]:
noisy_sampler = NeutralAtomSampler(noise=noise, seed=SEED)

cirq_result_noisy = noisy_sampler.run(ghz_cirq, repetitions=SHOTS)

# Accepted measurements: int8 2-D array, shape = (accepted_shots, n_qubits)
accepted_arr = cirq_result_noisy.measurements["result"]

# Raw measurements: str 2-D array, shape = (total_shots, n_qubits), "-" for loss
raw_arr = cirq_result_noisy.raw_measurements()["result"]

cirq_accepted = sum(1 for _ in accepted_arr)  # accepted_arr.shape[0]
cirq_total    = raw_arr.shape[0]
cirq_lost     = cirq_total - cirq_accepted

print(f"Total shots : {cirq_total}")
print(f"Accepted    : {cirq_accepted}  ({100*cirq_accepted/cirq_total:.1f}%)")
print(f"Lost        : {cirq_lost}  ({100*cirq_lost/cirq_total:.1f}%)")

# Build string-keyed counters from the raw array rows
def arr_to_counts(arr):
    return Counter("".join(row) for row in arr)

cirq_accepted_counts = arr_to_counts(accepted_arr.astype(str))
cirq_raw_counts      = arr_to_counts(raw_arr)

print("\nAccepted counts:", dict(cirq_accepted_counts))
print("Raw counts     :", dict(cirq_raw_counts))


## 9. Visualize Cirq Results — Filtered vs Raw Histograms

In [ ]:
def plot_counts(ax, counts, title):
    labels = sorted(counts.keys())
    values = [counts[k] for k in labels]
    bars = ax.bar(labels, values)
    ax.set_title(title)
    ax.set_xlabel("Outcome")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=45)
    # Dashed horizontal gridlines
    ax.yaxis.grid(True, linestyle="--", alpha=0.7)
    ax.set_axisbelow(True)
    # Count labels above each bar
    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            str(value),
            ha="center",
            va="bottom",
            fontsize=9,
        )

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_counts(axes[0], dict(cirq_accepted_counts), "Cirq — Accepted shots (loss filtered out)")
plot_counts(axes[1], dict(cirq_raw_counts),      "Cirq — Raw shots (loss bitstrings included)")
plt.tight_layout()
plt.show()


## 10. Side-by-Side Comparison — Qiskit vs Cirq

A 2×2 subplot grid comparing all four scenarios:

| | Accepted (filtered) | Raw (with loss) |
|---|---|---|
| **Qiskit** | top-left | top-right |
| **Cirq** | bottom-left | bottom-right |


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
fig.suptitle(
    f"NeutralAtomDevice — GHZ ({n_qubits} qubits, {SHOTS} shots, 8% Rz loss)",
    fontsize=14,
    fontweight="bold",
)

# Row 0: Qiskit
plot_counts(axes[0, 0], accepted,                 "Qiskit — Accepted")
plot_counts(axes[0, 1], raw,                      "Qiskit — Raw (loss included)")

# Row 1: Cirq
plot_counts(axes[1, 0], dict(cirq_accepted_counts), "Cirq — Accepted")
plot_counts(axes[1, 1], dict(cirq_raw_counts),      "Cirq — Raw (loss included)")

# Add framework labels on the left
for ax, label in zip(axes[:, 0], ["Qiskit", "Cirq"]):
    ax.set_ylabel(f"{label}\nCount")

plt.tight_layout()
plt.show()
